huggingface 라이브러리를 사용하여 embedding 추출하는 것을 목표로 함

huggingface LLM Course documentation을 적극 참고하는 중..

**갑자기 생각난건데 d_model 즉 히든dim를 4개로만 하면 4가지 속성에 대해서만 탐지하니까 걍 분류가 되지 않을까? 라는 상상을 함


모델 사용을 가장 쉽게 하기 위해선 pipeline() 함수를 활용할 수 있다.

전처리/후처리 과정을 모델과 연결시켜주는 역할!

즉, 원하는 모델 / 용도를 설정하면, 별도의 전처리/후처리 없이 바로 결과를 얻어낼 수 있게 한다.

굳이 따지면 우리 프로젝트는 후처리에서 분류 모델?을? 만든다고 해야 할까?


https://huggingface.co/docs/transformers/main/ko/tasks/zero_shot_image_classification


제로샷이라는게 파이프라인에 있는데 이건 라벨 정답 데이터가 없어도 레이블 달고 분류를 할 수 있다네? 어떻게???

아무튼 목표는 Multi-label Classification 임

근데 그럴려면 소프트맥스 말고 시그모이드를 쓰는게 나을거같음

 소프트맥스는 마지막 전체 클래스의 합이 1

 시그모이드는 그 부분에만 확률 0-1

 ReLU >??

 오차 계산(손실함수)도 클래스 개별로 봐야하니 bnary cross entropy

Cross-Entropy > Focal Loss

MSE

Log Loss

경사하강


 결정 경게(thresholding)도 그냥 0.5말고 캘리브레이션 해줘야 > MCC Threshold Optimization

In [3]:
!pip install -q transformers h5py

from huggingface_hub import login
#login()
from transformers import AutoModelForMaskedLM, AutoTokenizer
from google.colab import drive

import os
import tqdm
import torch
import numpy as np
import pandas as pd

In [4]:
drive.mount('/content/drive')
SAVE_PATH = '/content/drive/MyDrive/Github/Mprotein_hydrophobic/ESMC_embedding'
os.makedirs(SAVE_PATH, exist_ok=True)

model_id = 'Synthyra/ESMplusplus_large'
device = "cuda" if torch.cuda.is_available() else "cpu"

df = pd.read_csv('https://github.com/Rainbowbarkbark/Mprotein_hydrophobic/blob/main/Swissprot_Membrane_Train_Validation_dataset.csv?raw=true')

Mounted at /content/drive


In [ ]:
print(df.head())

   Unnamed: 0     ACC  Kingdom  Partition  Peripheral  Transmembrane  \
0           0  I3R9M8  Archaea          0           1              0   
1           1  I3R9M9  Archaea          1           1              0   
2           2  Q7ZAG8  Archaea          2           1              0   
3           3  Q8PZ67  Archaea          0           1              0   
4           4  Q9YGA6  Archaea          0           1              0   

   LipidAnchor  Soluble                                           Sequence  
0            0        0  MSTDSDAETVDLADGVDHQVAMVMDLNKCIGCQTCTVACKSLWTEG...  
1            0        0  MSRNDASQLDDGETTAESPPDDQANDAPEVGDPPGDPVDADSGVSR...  
2            0        0  MTKVLVLGGRFGALTAAYTLKRLVGSKADVKVINKSRFSYFRPALP...  
3            0        1  MPPKIAEVIQHDVCAACGACEAVCPIGAVTVKKAAEIRDPNDLSLY...  
4            0        0  MAGVRLVDVWKVFGEVTAVREMSLEVKDGEFMILLGPSGCGKTTTL...  


2. [h5py] 파일 열기('w') -> 빈 데이터셋 4개 생성 (embeddings, mask, labels, fold_ids)


3. [Loop] for 배치 in 데이터프레임:
    a. [Tokenizer] 시퀀스 -> 토큰화 (길이 1024 고정, Padding)
    b. [Model] 토큰 -> 모델 -> last_hidden_state (3D 벡터) 추출
    c. [Numpy] GPU 텐서 -> CPU 넘파이 변환 (float16)
    d. [h5py] 데이터셋 resize -> 데이터 밀어넣기

In [5]:
labels = ['Peripheral', 'Transmembrane', 'LipidAnchor', 'Soluble']
df_labels = df[labels].values.tolist()
df_partitions = df['Partition'].values.tolist()
df_sequences_raw = df['Sequence'].values.tolist()

model = AutoModelForMaskedLM.from_pretrained(model_id, trust_remote_code=True)
model = model.to(device).eval()
tokenizer = model.tokenizer
tokenized = tokenizer(df_sequences_raw[0], padding=True, return_tensors='pt')
tokenized = {key: val.to(device) for key, val in tokenized.items()}
with torch.no_grad():
  output = model(**tokenized, output_hidden_states=True)
print(output.last_hidden_state.shape)




#print(df_sequences_raw[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/771 [00:00<?, ?B/s]

modeling_esm_plusplus.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Synthyra/ESMplusplus_large:
- modeling_esm_plusplus.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/2.30G [00:00<?, ?B/s]

torch.Size([1, 354, 1152])


In [ ]:
#print(output.logits.shape) # language modeling logits, (batch_size, seq_len, vocab_size), (2, 11, 64)
print(output.last_hidden_state.shape) # last hidden state of the model, (batch_size, seq_len, hidden_size), (2, 11, 1152)
#print(output.loss) # language modeling loss if you passed labels

tensor([ 0, 20,  8, 11, 13,  8, 13,  5,  9, 11,  7, 13,  4,  5, 13,  6,  7, 13,
        21, 16,  7,  5, 20,  7, 20, 13,  4, 17, 15, 23, 12,  6, 23, 16, 11, 23,
        11,  7,  5, 23, 15,  8,  4, 22, 11,  9,  6,  6,  6, 10, 13, 19, 20, 19,
        22, 17, 17,  7,  9, 11, 15, 14,  6, 15,  6, 19, 14, 10, 17, 22,  9,  9,
         8,  6,  6,  6, 22, 15,  8,  8,  9, 21, 15,  9, 10, 15, 14,  6, 16, 12,
        14, 13, 15,  9, 13, 19,  6, 13,  5, 22,  9, 18, 17, 21,  9,  9, 12, 20,
        19, 17,  6,  8, 13, 10, 14,  4, 10, 14, 13,  8, 13, 14,  9, 22,  6, 14,
        17, 22, 13,  9, 13, 16,  6, 11,  6,  9, 19, 14, 17,  8, 19, 19, 18, 19,
         4, 14, 10, 12, 23, 17, 21, 23, 11, 21, 14,  8, 23,  7,  9,  5, 23, 14,
        10, 15,  5, 12, 19, 15, 10,  9,  9, 13,  6, 12,  7,  4, 12, 13, 16,  9,
        10, 23, 10,  6, 19, 10, 19, 23,  7,  9,  6, 23, 14, 19, 15, 15,  7, 19,
        19, 17,  5, 11, 16, 15, 11,  8,  9, 15, 23, 12, 18, 23, 19, 14, 10, 12,
         9,  6,  9,  6, 14, 13,  6, 15, 